# Householder-RoPE Colab Scaling Harness

## What this is
This notebook is a Colab harness for scaling Householder-RoPE on a realistic language-modeling workload. It drives the reusable runner in `scripts/run_real_data_scale_harness.py`, targets a single GPU by default, and switches to Flax on TPU when a TPU runtime is detected.

## Why it matters
The repository already verifies the Householder transport algebra and synthetic attention-block benchmarks. This notebook moves the test into a more realistic regime: tokenized `wikitext-103-raw-v1`, causal next-token prediction, larger model settings, and saved artifacts that are easy to compare across accelerators while exposing live optimization and RoPE-health telemetry.

## How to run / use it
1. Open the notebook in Colab and choose a GPU or TPU runtime.
2. Run the repo bootstrap cell. It clones `https://github.com/fyremael/HH.git` when the repo is not already present.
3. Adjust `COLAB_CONFIG` for your accelerator budget. The default profile is a 40GB A100-oriented capacity-limit run with a 4096-token context that is meant to push VRAM usage much harder than the earlier geometry-only profile.
4. Run the harness cell. The script emits ongoing stepwise metrics, per-variant JSONL and CSV histories, and multiple PNG diagnostics panels into `artifacts/`, and the notebook streams the child process logs line by line into the Colab cell output.
5. If you hit CUDA OOM, switch `ACTIVE_PROFILE` to `geometry_signal`, which remains the next best geometry-focused fallback.
6. Run the inspection cell to display the summary table, environment report, recent probe diagnostics, and saved plots.

## Validation plan
Validation here means three concrete checks: the package imports cleanly inside Colab, the realistic-data training loop completes without NaNs, the A100 profile uses materially more memory than the earlier small runs, and the ablation summary reports comparable losses and throughput for the requested reflector sweep while the diagnostics remain numerically well-behaved.

## Known failure modes
TPU runtimes may need one restart after installing `jax[tpu]`. `wikitext-103-raw-v1` is much larger than the smoke datasets used locally, so smaller GPUs may need lower `train_text_limit`, `seq_len`, or `batch_size`. If Colab loses its working directory, rerun the repo bootstrap cell before reinstalling dependencies. Dense diagnostics are intentionally limited by `diagnostic_token_limit` to keep the probe pass affordable at long context.

## Next steps
The next natural extensions are repeated-seed comparisons, throughput-vs-perplexity reports, deeper stacks with checkpointing, and a merge report that compares these realistic-data runs with the synthetic backend benchmarks already stored in `artifacts/`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def run_command(*command: str) -> None:
    print("+", " ".join(command))
    subprocess.run(list(command), check=True)

REPO_URL = "https://github.com/fyremael/HH.git"
REPO_CANDIDATES = [Path.cwd(), Path("/content/HH"), Path("/content/drive/MyDrive/HH")]
REPO_PATH = next((path for path in REPO_CANDIDATES if (path / "src" / "householder_rope").exists()), None)
if REPO_PATH is None:
    target = Path("/content/HH")
    if not target.exists():
        run_command("git", "clone", REPO_URL, str(target))
    REPO_PATH = target

ARTIFACT_DIR = REPO_PATH / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

run_command(sys.executable, "-m", "pip", "install", "-q", "uv")
run_command(
    "uv",
    "pip",
    "install",
    "--system",
    "datasets>=3.2.0",
    "transformers>=4.49.0",
    "sentencepiece>=0.2.0",
    "matplotlib>=3.9.0",
    "pandas>=2.2.0",
    "tqdm>=4.66.0",
)
if "COLAB_TPU_ADDR" in os.environ:
    run_command("uv", "pip", "install", "--system", "flax==0.12.5", "optax==0.2.6")
    try:
        import jax
        have_tpu = any(device.platform == "tpu" for device in jax.devices())
    except Exception:
        have_tpu = False
    if not have_tpu:
        run_command(
            "uv",
            "pip",
            "install",
            "--system",
            "jax[tpu]",
            "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        )
        raise SystemExit("Installed jax[tpu]. Restart the runtime once, then rerun the notebook from the top.")
run_command("uv", "pip", "install", "--system", "--no-deps", "-e", str(REPO_PATH))

print(f"Repo path: {REPO_PATH}")
print(f"Artifact directory: {ARTIFACT_DIR}")


In [ ]:
import json
import os

A100_PRESETS = {
    "fast_sanity": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 12000,
        "eval_text_limit": 1500,
        "seq_len": 256,
        "batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 1,
        "train_steps": 40,
        "eval_every": 10,
        "eval_batches": 6,
        "log_every": 5,
        "diagnostics_every": 10,
        "diagnostic_token_limit": 96,
        "num_layers": 2,
        "embed_dim": 512,
        "num_heads": 8,
        "mlp_ratio": 4.0,
        "learning_rate": 3.0e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 8],
        "output_stem": "colab_a100_fast_sanity",
    },
    "serious_comparison": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 60000,
        "eval_text_limit": 8000,
        "seq_len": 1024,
        "batch_size": 6,
        "eval_batch_size": 6,
        "gradient_accumulation_steps": 1,
        "train_steps": 300,
        "eval_every": 20,
        "eval_batches": 8,
        "log_every": 5,
        "diagnostics_every": 10,
        "diagnostic_token_limit": 192,
        "num_layers": 6,
        "embed_dim": 1024,
        "num_heads": 16,
        "mlp_ratio": 4.0,
        "learning_rate": 2.5e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 8, 16],
        "output_stem": "colab_a100_40gb_default",
    },
    "geometry_signal": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 80000,
        "eval_text_limit": 10000,
        "seq_len": 2048,
        "batch_size": 3,
        "eval_batch_size": 3,
        "gradient_accumulation_steps": 1,
        "train_steps": 600,
        "eval_every": 25,
        "eval_batches": 8,
        "log_every": 10,
        "diagnostics_every": 25,
        "diagnostic_token_limit": 256,
        "num_layers": 6,
        "embed_dim": 1024,
        "num_heads": 16,
        "mlp_ratio": 4.0,
        "learning_rate": 2.0e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 16, 32],
        "output_stem": "colab_a100_geometry_signal",
    },
    "capacity_limit": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 120000,
        "eval_text_limit": 12000,
        "seq_len": 4096,
        "batch_size": 4,
        "eval_batch_size": 4,
        "gradient_accumulation_steps": 1,
        "train_steps": 400,
        "eval_every": 25,
        "eval_batches": 6,
        "log_every": 5,
        "diagnostics_every": 25,
        "diagnostic_token_limit": 256,
        "num_layers": 12,
        "embed_dim": 1536,
        "num_heads": 24,
        "mlp_ratio": 4.0,
        "learning_rate": 1.5e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 16, 32],
        "output_stem": "colab_a100_capacity_limit",
    },
    "long_context_stress": {
        "backend": "auto",
        "dataset_name": "wikitext",
        "dataset_config": "wikitext-103-raw-v1",
        "tokenizer_name": "gpt2",
        "train_text_limit": 50000,
        "eval_text_limit": 6000,
        "seq_len": 1024,
        "batch_size": 4,
        "eval_batch_size": 4,
        "gradient_accumulation_steps": 1,
        "train_steps": 120,
        "eval_every": 20,
        "eval_batches": 6,
        "log_every": 10,
        "diagnostics_every": 20,
        "diagnostic_token_limit": 192,
        "num_layers": 4,
        "embed_dim": 768,
        "num_heads": 12,
        "mlp_ratio": 4.0,
        "learning_rate": 2.0e-4,
        "weight_decay": 1.0e-2,
        "seed": 17,
        "use_compile": True,
        "use_bf16": True,
        "householder_init": "jittered_pairs",
        "reflector_sweep": [0, 8, 16],
        "output_stem": "colab_a100_long_context_stress",
    },
}

ACTIVE_PROFILE = "capacity_limit"
COLAB_CONFIG = dict(A100_PRESETS[ACTIVE_PROFILE])

BACKEND = COLAB_CONFIG["backend"]
if BACKEND == "auto":
    BACKEND = "flax" if "COLAB_TPU_ADDR" in os.environ else "torch"

print("Available profiles:", ", ".join(A100_PRESETS))
print("Selected profile:", ACTIVE_PROFILE)
print("Selected backend:", BACKEND)
print("TPU runtime hint:", "COLAB_TPU_ADDR" in os.environ)
print("If this profile OOMs, switch ACTIVE_PROFILE to 'geometry_signal'.")
print(json.dumps(COLAB_CONFIG, indent=2))


In [ ]:
import os
import subprocess
import sys

command = [
    sys.executable,
    "-u",
    str(REPO_PATH / "scripts" / "run_real_data_scale_harness.py"),
    "--backend",
    BACKEND,
    "--dataset-name",
    COLAB_CONFIG["dataset_name"],
    "--dataset-config",
    COLAB_CONFIG["dataset_config"],
    "--tokenizer-name",
    COLAB_CONFIG["tokenizer_name"],
    "--train-text-limit",
    str(COLAB_CONFIG["train_text_limit"]),
    "--eval-text-limit",
    str(COLAB_CONFIG["eval_text_limit"]),
    "--seq-len",
    str(COLAB_CONFIG["seq_len"]),
    "--batch-size",
    str(COLAB_CONFIG["batch_size"]),
    "--eval-batch-size",
    str(COLAB_CONFIG["eval_batch_size"]),
    "--gradient-accumulation-steps",
    str(COLAB_CONFIG["gradient_accumulation_steps"]),
    "--train-steps",
    str(COLAB_CONFIG["train_steps"]),
    "--eval-every",
    str(COLAB_CONFIG["eval_every"]),
    "--eval-batches",
    str(COLAB_CONFIG["eval_batches"]),
    "--log-every",
    str(COLAB_CONFIG["log_every"]),
    "--diagnostics-every",
    str(COLAB_CONFIG["diagnostics_every"]),
    "--diagnostic-token-limit",
    str(COLAB_CONFIG["diagnostic_token_limit"]),
    "--num-layers",
    str(COLAB_CONFIG["num_layers"]),
    "--embed-dim",
    str(COLAB_CONFIG["embed_dim"]),
    "--num-heads",
    str(COLAB_CONFIG["num_heads"]),
    "--mlp-ratio",
    str(COLAB_CONFIG["mlp_ratio"]),
    "--learning-rate",
    str(COLAB_CONFIG["learning_rate"]),
    "--weight-decay",
    str(COLAB_CONFIG["weight_decay"]),
    "--seed",
    str(COLAB_CONFIG["seed"]),
    "--householder-init",
    COLAB_CONFIG["householder_init"],
    "--reflector-sweep",
    *[str(value) for value in COLAB_CONFIG["reflector_sweep"]],
    "--output-stem",
    COLAB_CONFIG["output_stem"],
    "--log-level",
    "INFO",
]
if COLAB_CONFIG["use_compile"]:
    command.append("--use-compile")
else:
    command.append("--no-use-compile")
if COLAB_CONFIG["use_bf16"]:
    command.append("--use-bf16")
else:
    command.append("--no-use-bf16")

print("Live harness logs will stream below.", flush=True)
print("+", " ".join(command), flush=True)
env = dict(os.environ, PYTHONUNBUFFERED="1")
process = subprocess.Popen(
    command,
    cwd=REPO_PATH,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

stem = COLAB_CONFIG["output_stem"]
metrics_path = ARTIFACT_DIR / f"{stem}_metrics.json"
summary_path = ARTIFACT_DIR / f"{stem}_summary.csv"
loss_curve_path = ARTIFACT_DIR / f"{stem}_loss_curves.png"
throughput_path = ARTIFACT_DIR / f"{stem}_throughput.png"
dynamics_path = ARTIFACT_DIR / f"{stem}_training_dynamics.png"
component_path = ARTIFACT_DIR / f"{stem}_component_diagnostics.png"
rope_path = ARTIFACT_DIR / f"{stem}_rope_diagnostics.png"

payload = json.loads(metrics_path.read_text(encoding="utf-8"))
summary_df = pd.read_csv(summary_path)

display(summary_df)
print(json.dumps(payload["environment"], indent=2))
print(json.dumps(payload["dataset_summary"], indent=2))
for result in payload["results"]:
    print(result["variant"], json.dumps(result.get("latest_probe_summary"), indent=2))
    print("history_jsonl_path:", result["history_jsonl_path"])
    print("history_csv_path:", result["history_csv_path"])
display(Image(filename=str(loss_curve_path)))
display(Image(filename=str(throughput_path)))
display(Image(filename=str(dynamics_path)))
display(Image(filename=str(component_path)))
display(Image(filename=str(rope_path)))
